# Synth-ID Text detectability and paraphrasing attack robustness
The code in this notebook is mainly taken from google's [synthid-text repository](https://github.com/google-deepmind/synthid-text)

## Install and import the required Python packages
Running this cell may require you to restart your session.

In [ ]:

! pip install synthid-text[notebook]

In [ ]:
from collections.abc import Sequence
import enum
import gc

import datasets
import huggingface_hub


In [ ]:
import torch
print(torch.cuda.is_available())       # Should be True
print(torch.cuda.get_device_name(0))   # Shows GPU name
print(torch.cuda.device_count())    

In [ ]:
from synthid_text import detector_mean
from synthid_text import logits_processing
from synthid_text import synthid_mixin
from synthid_text import detector_bayesian
import tensorflow as tf
import torch
import tqdm
import transformers

## Choose model and set device and config

In [ ]:
class ModelName(enum.Enum):
  GPT2 = 'gpt2'
  GEMMA_2B = 'google/gemma-2b-it'
  GEMMA_7B = 'google/gemma-7b-it'
MODEL_NAME = ModelName('google/gemma-2b-it')
if MODEL_NAME is not ModelName.GPT2:
  huggingface_hub.notebook_login()

In [ ]:
DEVICE = (
    torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
)
DEVICE

In [ ]:
CONFIG = synthid_mixin.DEFAULT_WATERMARKING_CONFIG

## Initialize the required constants, tokenizer, and logits processor

In [ ]:

BATCH_SIZE = 8
NUM_BATCHES = 320
OUTPUTS_LEN = 1024
TEMPERATURE = 0.5
TOP_K = 40
TOP_P = 0.99

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME.value)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

logits_processor = logits_processing.SynthIDLogitsProcessor(
    **CONFIG, top_k=TOP_K, temperature=TEMPERATURE
)

## Utility functions to load models

In [ ]:


def load_model(
    model_name: ModelName,
    expected_device: torch.device,
    enable_watermarking: bool = False,
) -> transformers.PreTrainedModel:
  if model_name == ModelName.GPT2:
    model_cls = (
        synthid_mixin.SynthIDGPT2LMHeadModel
        if enable_watermarking
        else transformers.GPT2LMHeadModel
    )
    model = model_cls.from_pretrained(model_name.value, device_map='auto')
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
  else:
    model_cls = (
        synthid_mixin.SynthIDGemmaForCausalLM
        if enable_watermarking
        else transformers.GemmaForCausalLM
    )
    model = model_cls.from_pretrained(
        model_name.value,
        device_map='auto',
        torch_dtype=torch.bfloat16,
    )

  if str(model.device) != str(expected_device):
    raise ValueError('Model device not as expected.')

  return model


## Constants

In [ ]:


NUM_NEGATIVES = 10000
POS_BATCH_SIZE = 32
NUM_POS_BATCHES = 313
NEG_BATCH_SIZE = 32
# Truncate outputs to this length for training.
POS_TRUNCATION_LENGTH = 200
NEG_TRUNCATION_LENGTH = 200
# Pad trucated outputs to this length for equal shape across all batches.
MAX_PADDED_LENGTH = 1000
TEMPERATURE = 1.0

## Generate model responses and compute g-values

In [ ]:


def generate_responses(example_inputs, enable_watermarking):
  inputs = tokenizer(
      example_inputs,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)

  # @title Watermarked output preparation for detector training
  gc.collect()
  torch.cuda.empty_cache()

  model = load_model(
      MODEL_NAME,
      expected_device=DEVICE,
      enable_watermarking=enable_watermarking,
  )
  torch.manual_seed(0)
  _, inputs_len = inputs['input_ids'].shape

  outputs = model.generate(
      **inputs,
      do_sample=True,
      max_length=inputs_len + OUTPUTS_LEN,
      temperature=TEMPERATURE,
      top_k=TOP_K,
      top_p=TOP_P,
  )

  outputs = outputs[:, inputs_len:]
  decoded_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True) 

  # eos mask is computed, skip first ngram_len - 1 tokens
  # eos_mask will be of shape [batch_size, output_len]
  eos_token_mask = logits_processor.compute_eos_token_mask(
      input_ids=outputs,
      eos_token_id=tokenizer.eos_token_id,
  )[:, CONFIG['ngram_len'] - 1 :]

  # context repetition mask is computed
  context_repetition_mask = logits_processor.compute_context_repetition_mask(
      input_ids=outputs,
  )
  # context repitition mask shape [batch_size, output_len - (ngram_len - 1)]

  combined_mask = context_repetition_mask * eos_token_mask

  g_values = logits_processor.compute_g_values(
      input_ids=outputs,
  )
  # g values shape [batch_size, output_len - (ngram_len - 1), depth]

  return g_values, combined_mask, decoded_texts
    
example_inputs = [
    'Once upon a time, in a land of magic, there lived a beautiful princess named Aurora.',
    'I enjoy walking with my cute dog',
    'I am from New York'
]

wm_g_values, wm_mask, wm_texts = generate_responses(example_inputs, enable_watermarking=True)
uwm_g_values, uwm_mask, uwm_texts = generate_responses(example_inputs, enable_watermarking=False)

for i, prompt in enumerate(example_inputs):
    print(f"Prompt: {prompt}")
    print(f"  Watermarked:   {wm_texts[i]}")
    print(f"  Unwatermarked: {uwm_texts[i]}")
    print()


## Get Mean detector scores for the generated outputs

In [ ]:


# Watermarked responses tend to have higher Mean scores than unwatermarked
# responses. To classify responses you can set a score threshold, but this will
# depend on the distribution of scores for your use-case and your desired false
# positive / false negative rates.

wm_mean_scores = detector_mean.mean_score(
    wm_g_values.cpu().numpy(), wm_mask.cpu().numpy()
)
uwm_mean_scores = detector_mean.mean_score(
    uwm_g_values.cpu().numpy(), uwm_mask.cpu().numpy()
)

print('Mean scores for watermarked responses: ', wm_mean_scores)
print('Mean scores for unwatermarked responses: ', uwm_mean_scores)

# You may find that the Weighted Mean scoring function gives better
# classification performance than the Mean scoring function (in particular,
# higher scores for watermarked responses). See the paper for full details.

wm_weighted_mean_scores = detector_mean.weighted_mean_score(
    wm_g_values.cpu().numpy(), wm_mask.cpu().numpy()
)
uwm_weighted_mean_scores = detector_mean.weighted_mean_score(
    uwm_g_values.cpu().numpy(), uwm_mask.cpu().numpy()
)

print(
    'Weighted Mean scores for watermarked responses: ', wm_weighted_mean_scores
)
print(
    'Weighted Mean scores for unwatermarked responses: ',
    uwm_weighted_mean_scores,
)

In [ ]:
! pip install nltk

In [ ]:
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')

# helpers

def get_synonym(word, pos_tag):
    """Return a simple synonym from WordNet, or None if not found."""
    wn_pos = None
    if pos_tag.startswith('NN'):   wn_pos = wordnet.NOUN
    elif pos_tag.startswith('VB'): wn_pos = wordnet.VERB
    elif pos_tag.startswith('JJ'): wn_pos = wordnet.ADJ
    elif pos_tag.startswith('RB'): wn_pos = wordnet.ADV

    if wn_pos is None:
        return None

    synsets = wordnet.synsets(word, pos=wn_pos)
    for synset in synsets:
        for lemma in synset.lemmas():
            candidate = lemma.name().replace('_', ' ')
            if candidate.lower() != word.lower():
                return candidate
    return None


def compute_watermark_score(text):
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    input_ids = inputs['input_ids']

    eos_mask = logits_processor.compute_eos_token_mask(
        input_ids=input_ids,
        eos_token_id=tokenizer.eos_token_id,
    )[:, CONFIG['ngram_len'] - 1:]

    context_rep_mask = logits_processor.compute_context_repetition_mask(
        input_ids=input_ids,
    )

    combined_mask = context_rep_mask * eos_mask  # [batch, seq_len]

    g_values = logits_processor.compute_g_values(
        input_ids=input_ids,
    ).float()  # cast to float here

    # average over depth first → [batch, seq_len]
    g_mean_depth = g_values.mean(dim=-1)

    # apply mask and average over valid positions only
    masked_sum = (g_mean_depth * combined_mask.float()).sum()
    valid_positions = combined_mask.sum().clamp(min=1).float()
    score = masked_sum / valid_positions

    return score.item()

# paraphrasing attack

def paraphrasing_attack(watermarked_text, detector=None, threshold=0.5, verbose=True):
    """
    Replace words one by one with synonyms and track whether
    the watermark score drops below `threshold`.

    Returns a dict with:
      - scores_per_step  : watermark score after each substitution
      - modified_texts   : text after each substitution
      - substitutions    : list of (position, original, replacement)
      - first_broken_at  : index at which watermark first drops below threshold
    """
    tokens = nltk.word_tokenize(watermarked_text)
    pos_tags = nltk.pos_tag(tokens)

    current_tokens = tokens.copy()
    scores         = []
    modified_texts = []
    substitutions  = []

    initial_score = compute_watermark_score(watermarked_text)
    scores.append(initial_score)
    modified_texts.append(watermarked_text)

    if verbose:
        print(f"Original text:  {watermarked_text}")
        print(f"Initial score:  {initial_score:.4f}\n")

    first_broken_at = None

    for i, (word, pos) in enumerate(pos_tags):
        synonym = get_synonym(word, pos)
        if synonym is None:
            continue  # skip words with no synonym

        prev_word           = current_tokens[i]
        current_tokens[i]   = synonym
        new_text            = ' '.join(current_tokens)
        new_score           = compute_watermark_score(new_text)

        scores.append(new_score)
        modified_texts.append(new_text)
        substitutions.append((i, prev_word, synonym))

        if verbose:
            print(f"Step {len(substitutions):>3} | "
                  f"'{prev_word}' → '{synonym}' | "
                  f"score: {new_score:.4f} "
                  f"{'✗ watermark broken' if new_score < threshold else '✓'}")

        if first_broken_at is None and new_score < threshold:
            first_broken_at = len(substitutions)

    if verbose:
        print(f"\nWatermark first broken at step: {first_broken_at}")
        print(f"Final text: {' '.join(current_tokens)}")

    return {
        'scores_per_step'  : scores,
        'modified_texts'   : modified_texts,
        'substitutions'    : substitutions,
        'first_broken_at'  : first_broken_at,
    }

## Run attack - with 0.50 threshold

In [ ]:
# run attack on all watermarked outputs 
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
results = []
for i, wm_text in enumerate(wm_texts):
    print(f"\n{'='*60}")
    print(f"Prompt: {example_inputs[i]}")
    print(f"{'='*60}")
    result = paraphrasing_attack(wm_text, threshold=0.52)
    results.append(result)

In [ ]:
!pip install matplotlib

## Visualise score decay - with 0.50 threshold

In [ ]:
# visualise score decay 
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(wm_texts), figsize=(15, 4), sharey=True)

for i, (result, ax) in enumerate(zip(results, axes)):
    ax.plot(result['scores_per_step'], marker='o', markersize=4, color='#185FA5')
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='threshold')
    if result['first_broken_at'] is not None:
        ax.axvline(x=result['first_broken_at'], color='orange',
                   linestyle=':', linewidth=1.5, label='watermark broken')
    ax.set_title(f'Text {i+1}', fontsize=11)
    ax.set_xlabel('Substitution step')
    ax.set_ylabel('Watermark score')
    ax.set_ylim(0.45, 0.65)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Paraphrasing Attack — Watermark Score Decay', fontsize=13)
plt.tight_layout()
plt.savefig('paraphrasing_attack.png', dpi=150)
plt.show()

## Run attack - with 0.52 threshold

In [ ]:
# Set threshold to 0.52, which is the value we chose from the first experiment 
results = []
for i, wm_text in enumerate(wm_texts):
    print(f"\n{'='*60}")
    print(f"Prompt: {example_inputs[i]}")
    print(f"{'='*60}")
    result = paraphrasing_attack(wm_text, threshold=0.52)
    results.append(result)

## Visualise score decay - with 0.52 threshold

In [ ]:


import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(wm_texts), figsize=(15, 4), sharey=True)

for i, (result, ax) in enumerate(zip(results, axes)):
    ax.plot(result['scores_per_step'], marker='o', markersize=4, color='#185FA5')
    ax.axhline(y=0.52, color='red', linestyle='--', linewidth=1, label='threshold')
    if result['first_broken_at'] is not None:
        ax.axvline(x=result['first_broken_at'], color='orange',
                   linestyle=':', linewidth=1.5, label='watermark broken')
    ax.set_title(f'Text {i+1}', fontsize=11)
    ax.set_xlabel('Substitution step')
    ax.set_ylabel('Watermark score')
    ax.set_ylim(0.45, 0.65)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Paraphrasing Attack — Watermark Score Decay', fontsize=13)
plt.tight_layout()
plt.savefig('paraphrasing_attack.png', dpi=150)
plt.show()

# (Not ran) code to train the Bayesian classifier 

## Generate watermarked samples for training Bayesian detector

In [ ]:

gc.collect()
torch.cuda.empty_cache()

model = load_model(MODEL_NAME, expected_device=DEVICE, enable_watermarking=True)
torch.manual_seed(0)

eli5_prompts = datasets.load_dataset("Pavithree/eli5")

wm_outputs = []

for batch_id in tqdm.tqdm(range(NUM_POS_BATCHES)):
  prompts = eli5_prompts['train']['title'][
      batch_id * POS_BATCH_SIZE:(batch_id + 1) * POS_BATCH_SIZE]
  prompts = [_process_raw_prompt(prompt.encode()) for prompt in prompts]
  inputs = tokenizer(
      prompts,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)
  _, inputs_len = inputs['input_ids'].shape

  outputs = model.generate(
      **inputs,
      do_sample=True,
      max_length=inputs_len + OUTPUTS_LEN,
      temperature=TEMPERATURE,
      top_k=TOP_K,
      top_p=TOP_P,
  )

  wm_outputs.append(outputs[:, inputs_len:])

  del outputs, inputs, prompts

del model
gc.collect()
torch.cuda.empty_cache()

## Generate unwatermarked samples for training Bayesian detector

In [ ]:

dataset, info = tfds.load('wikipedia/20230601.en', split='train', with_info=True)

dataset = dataset.take(10000)

# Convert the dataset to a DataFrame
df = tfds.as_dataframe(dataset, info)
ds = tf.data.Dataset.from_tensor_slices(dict(df))
tf.random.set_seed(0)
ds = ds.shuffle(buffer_size=10_000)
ds = ds.batch(batch_size=1)

tokenized_uwm_outputs = []
lengths = []
batched = []
# Pad to this length (on the right) for batching.
padded_length = 2500
for i, batch in tqdm.tqdm(enumerate(ds)):
  responses = [val.decode() for val in batch['text'].numpy()]
  inputs = tokenizer(
      responses,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)
  line = inputs['input_ids'].cpu().numpy()[0].tolist()
  if len(line) >= padded_length:
    line = line[:padded_length]
  else:
    line = line + [
        tokenizer.eos_token_id for _ in range(padded_length - len(line))
    ]
  batched.append(torch.tensor(line, dtype=torch.long, device=DEVICE)[None, :])
  if len(batched) == NEG_BATCH_SIZE:
    tokenized_uwm_outputs.append(torch.cat(batched, dim=0))
    batched = []
  if i > NUM_NEGATIVES:
    break

## Train the Bayesian detector

In [ ]:

bayesian_detector, test_loss = (
    detector_bayesian.BayesianDetector.train_best_detector(
        tokenized_wm_outputs=wm_outputs,
        tokenized_uwm_outputs=tokenized_uwm_outputs,
        logits_processor=logits_processor,
        tokenizer=tokenizer,
        torch_device=DEVICE,
        max_padded_length=MAX_PADDED_LENGTH,
        pos_truncation_length=POS_TRUNCATION_LENGTH,
        neg_truncation_length=NEG_TRUNCATION_LENGTH,
        verbose=True,
        learning_rate=3e-3,
        n_epochs=100,
        l2_weights=np.zeros((1,)),
    )
)

## Get Bayesian detector scores for the generated outputs.

In [ ]:
# Watermarked responses tend to have higher Bayesian scores than unwatermarked
# responses. To classify responses you can set a score threshold, but this will
# depend on the distribution of scores for your use-case and your desired false
# positive / false negative rates. 

wm_bayesian_scores = bayesian_detector.score(
    wm_g_values.cpu().numpy(), wm_mask.cpu().numpy()
)
uwm_bayesian_scores = bayesian_detector.score(
    uwm_g_values.cpu().numpy(), uwm_mask.cpu().numpy()
)

print('Bayesian scores for watermarked responses: ', wm_bayesian_scores)
print('Bayesian scores for unwatermarked responses: ', uwm_bayesian_scores)